# Task 2.2: Reproduction of Core Contribution

**Paper:** Improving the Fisher Kernel for Large-Scale Image Classification (Perronnin et al., ECCV 2010)  
**Student:** Prince Sahoo (230060)

## Contribution Being Reproduced

We reproduce the **Improved Fisher Vector (IFK)** pipeline from Section 3 of the paper — specifically the combination of **power normalization** (Section 3.2, Eq. 15) and **L2 normalization** (Section 3.1, Eq. 14) applied to GMM-based Fisher Vectors. We compare the standard Fisher Kernel (no normalization) against variants with each normalization alone and both combined, mirroring the ablation in Table 1 of the paper.

**Evaluation metric:** Mean Average Precision (mAP), the same metric used throughout the paper (Section 4.2). For the multi-class Olivetti Faces dataset, we compute per-class Average Precision from the SVM's decision function scores and average across all classes.

In [1]:
# ============================================================
# IMPORTS AND ALL HYPERPARAMETERS (defined in one place)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.svm import LinearSVC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ---- Reproducibility ----
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---- Dataset ----
TEST_SIZE = 0.5               # 50/50 split — only 5 train/class for challenging setup

# ---- Patch Extraction (Section 4.1) ----
PATCH_SIZE = 8          # Paper uses 32x32; we use 8x8 for toy dataset
GRID_STEP = 6           # Sparser grid → fewer patches → more visible sparsity
NOISE_STD = 0.05        # Small noise for descriptor variability

# ---- PCA (Section 4.1) ----
N_PCA_COMPONENTS = 32   # Paper reduces to 64-D; we use 32-D

# ---- GMM (Section 2) ----
N_GAUSSIANS = 16        # Paper uses K=256; we use K=16 for toy data

# ---- Power Normalization (Section 3.2, Eq. 15) ----
ALPHA = 0.5             # Same as paper

# ---- SVM (Section 4.1) ----
SVM_C = 1.0
SVM_MAX_ITER = 10000

print("All hyperparameters defined.")
print(f"  K={N_GAUSSIANS}, D_pca={N_PCA_COMPONENTS}, alpha={ALPHA}")
print(f"  Fisher Vector dimensionality: 2 * {N_GAUSSIANS} * {N_PCA_COMPONENTS} = {2 * N_GAUSSIANS * N_PCA_COMPONENTS}")

All hyperparameters defined.
  K=16, D_pca=32, alpha=0.5
  Fisher Vector dimensionality: 2 * 16 * 32 = 1024


All hyperparameters are defined in one place for reproducibility. We use K=16 Gaussians instead of the paper's K=256 because our training set has ~20,000 patches total — K=256 would give each Gaussian only ~78 patches on average, too few for stable estimation. The paper uses K=256 on millions of descriptors.

In [2]:
# ============================================================
# STEP 1: Load dataset and extract patches
# (Same setup as Task 2.1 — included for self-containment)
# ============================================================
data = fetch_olivetti_faces(shuffle=False)
images = data.images
targets = data.target

X_train_img, X_test_img, y_train, y_test = train_test_split(
    images, targets,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=targets
)

def extract_patches(image, patch_size=PATCH_SIZE, step=GRID_STEP, noise_std=NOISE_STD):
    """Extract dense overlapping patches from a single image."""
    h, w = image.shape
    patches = []
    for y in range(0, h - patch_size + 1, step):
        for x in range(0, w - patch_size + 1, step):
            patch = image[y:y+patch_size, x:x+patch_size].flatten()
            if noise_std > 0:
                patch = patch + np.random.randn(*patch.shape) * noise_std
            patches.append(patch)
    return np.array(patches)

np.random.seed(RANDOM_SEED)
train_patches_per_image = [extract_patches(img) for img in X_train_img]
test_patches_per_image = [extract_patches(img) for img in X_test_img]
all_train_patches = np.vstack(train_patches_per_image)

print(f"Training: {len(train_patches_per_image)} images, {all_train_patches.shape[0]:,} total patches")
print(f"Test: {len(test_patches_per_image)} images")
print(f"Patches per image: ~{len(train_patches_per_image[0])}")
print(f"Patch dimensionality: {all_train_patches.shape[1]}")

Training: 200 images, 20,000 total patches
Test: 200 images
Patches per image: ~100
Patch dimensionality: 64


We load the Olivetti Faces dataset with a 50/50 stratified split (200 train, 200 test — only 5 images per class for training). Dense 8×8 patches are extracted on a 6-pixel grid with small noise added. This is our simplified version of the paper's SIFT descriptor extraction (Section 4.1), where they extract 128-D SIFT from 32×32 patches every 16 pixels at five scales.

In [3]:
# ============================================================
# STEP 2: PCA Dimensionality Reduction
# Paper Section 4.1: "Both SIFT and color features are
# reduced to 64 dimensions using PCA."
# ============================================================
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_SEED)
pca.fit(all_train_patches)

train_patches_pca = [pca.transform(p) for p in train_patches_per_image]
test_patches_pca = [pca.transform(p) for p in test_patches_per_image]
all_train_pca = np.vstack(train_patches_pca)

explained_var = pca.explained_variance_ratio_.sum()
print(f"PCA: {all_train_patches.shape[1]}-D → {N_PCA_COMPONENTS}-D")
print(f"Explained variance retained: {explained_var:.1%}")

PCA: 64-D → 32-D
Explained variance retained: 95.3%


PCA reduces each patch descriptor from 64 to 32 dimensions, retaining most of the variance while halving the Fisher Vector dimensionality. The paper does the same reduction (128-D → 64-D) to keep the representation manageable and to decorrelate features before fitting the diagonal-covariance GMM (Section 4.1). PCA is fit on training patches only.

In [4]:
# ============================================================
# STEP 3: GMM Training
# Paper Section 2: "We follow [22] and choose u_lambda to be
# a Gaussian mixture model (GMM)."
# Eq. 6: gamma_t(i) = w_i * u_i(x_t) / sum_j w_j * u_j(x_t)
# ============================================================
gmm = GaussianMixture(
    n_components=N_GAUSSIANS,
    covariance_type='diag',    # Paper: "We assume diagonal covariance" (Section 2)
    max_iter=200,
    random_state=RANDOM_SEED,
    n_init=3
)
gmm.fit(all_train_pca)

print(f"GMM trained with K={N_GAUSSIANS} Gaussians on {all_train_pca.shape[0]:,} descriptors")
print(f"Covariance type: diagonal (Section 2)")
print(f"Converged: {gmm.converged_}")

GMM trained with K=16 Gaussians on 20,000 descriptors
Covariance type: diagonal (Section 2)
Converged: True


We train a GMM with K=16 Gaussians and **diagonal covariance matrices** on all PCA-reduced training patches. The GMM serves as the universal "visual vocabulary" that models the distribution of all possible local descriptors (Section 2). We use `covariance_type='diag'` because the paper explicitly assumes diagonal covariances: *"We assume that the covariance matrices are diagonal"* (Section 2). The paper uses K=256 trained via EM on millions of descriptors.

In [5]:
# ============================================================
# STEP 4: Fisher Vector Computation
# Paper Section 2, Equations 7 and 8:
#   G_mu,i   = (1 / T*sqrt(w_i)) * sum_t gamma_t(i) * (x_t - mu_i) / sigma_i
#   G_sigma,i = (1 / T*sqrt(2*w_i)) * sum_t gamma_t(i) * [(x_t-mu_i)^2/sigma_i^2 - 1]
# ============================================================
def compute_fisher_vectors(patches_per_image, gmm):
    """
    Compute Fisher Vectors for a list of images.
    Implements Equations 5, 7, and 8 from the paper.
    
    For each image with T descriptors:
    - Compute soft assignment gamma_t(i) via Eq. 6
    - Compute mean gradient G_mu,i via Eq. 7
    - Compute variance gradient G_sigma,i via Eq. 8
    - Concatenate all: 2*K*D dimensional vector
    """
    K = gmm.n_components
    D = gmm.means_.shape[1]
    weights = gmm.weights_                # w_i, shape (K,)
    means = gmm.means_                    # mu_i, shape (K, D)
    covars = gmm.covariances_             # sigma_i^2, shape (K, D) — diagonal
    
    fisher_vectors = []
    
    for X in patches_per_image:
        T = len(X)
        
        # Eq. 6: Soft assignment (posterior probability)
        gamma = gmm.predict_proba(X)  # shape (T, K)
        
        fv_parts = []
        for i in range(K):
            gamma_i = gamma[:, i:i+1]  # (T, 1)
            
            # Eq. 7: Gradient w.r.t. mean mu_i
            diff = (X - means[i]) / np.sqrt(covars[i])
            g_mu = (1.0 / (T * np.sqrt(weights[i]))) * np.sum(gamma_i * diff, axis=0)
            
            # Eq. 8: Gradient w.r.t. standard deviation sigma_i
            diff_sq = (X - means[i]) ** 2 / covars[i] - 1
            g_sigma = (1.0 / (T * np.sqrt(2 * weights[i]))) * np.sum(gamma_i * diff_sq, axis=0)
            
            fv_parts.append(g_mu)
            fv_parts.append(g_sigma)
        
        fv = np.concatenate(fv_parts)
        fisher_vectors.append(fv)
    
    return np.array(fisher_vectors)


train_fv = compute_fisher_vectors(train_patches_pca, gmm)
test_fv = compute_fisher_vectors(test_patches_pca, gmm)

print(f"Fisher Vector shape: {train_fv.shape}")
print(f"  = 2 × K({N_GAUSSIANS}) × D({N_PCA_COMPONENTS}) = {2 * N_GAUSSIANS * N_PCA_COMPONENTS}")
print(f"Training FVs: {train_fv.shape[0]}, Test FVs: {test_fv.shape[0]}")
print(f"FV value range: [{train_fv.min():.4f}, {train_fv.max():.4f}]")
print(f"Sparsity (% values |z| < 0.001): {(np.abs(train_fv) < 0.001).mean():.1%}")

Fisher Vector shape: (200, 1024)
  = 2 × K(16) × D(32) = 1024
Training FVs: 200, Test FVs: 200
FV value range: [-1.0002, 1.6313]
Sparsity (% values |z| < 0.001): 3.8%


This is the core Fisher Vector computation implementing Equations 7 and 8. For each image, we compute the soft assignment $\gamma_t(i)$ of each descriptor to each Gaussian (Eq. 6), then the **mean gradient** $\mathcal{G}_{\mu,i}^X$ that captures how descriptors shift from each Gaussian's center (Eq. 7), and the **variance gradient** $\mathcal{G}_{\sigma,i}^X$ that captures whether they are more or less spread out than expected (Eq. 8). The final vector is $2KD = 1{,}024$-dimensional. Note the reported sparsity — this is the problem that power normalization addresses.

In [6]:
# ============================================================
# STEP 5: Power Normalization
# Paper Section 3.2, Equation 15:
#   f(z) = sign(z) * |z|^alpha, with alpha = 0.5
# ============================================================
def power_normalize(X, alpha=ALPHA):
    """Apply element-wise power normalization (Eq. 15)."""
    return np.sign(X) * np.abs(X) ** alpha


train_fv_pn = power_normalize(train_fv)
test_fv_pn = power_normalize(test_fv)

print(f"Power normalization applied with alpha={ALPHA}")
print(f"Before PN — sparsity (|z| < 0.001): {(np.abs(train_fv) < 0.001).mean():.1%}")
print(f"After  PN — sparsity (|z| < 0.001): {(np.abs(train_fv_pn) < 0.001).mean():.1%}")

Power normalization applied with alpha=0.5


Before PN — sparsity (|z| < 0.001): 3.8%
After  PN — sparsity (|z| < 0.001): 0.2%


Power normalization applies $f(z) = \text{sign}(z) \cdot |z|^{0.5}$ element-wise to the Fisher Vector (Eq. 15). Section 3.2 explains that as K grows, the Fisher Vector becomes sparse (most values near zero — see Figure 1), making the dot-product a poor similarity metric. Power normalization "un-sparsifies" the distribution. The paper reports this is the single most impactful improvement (+6.3% AP alone on PASCAL VOC 2007, Table 1).

In [7]:
# ============================================================
# STEP 6: L2 Normalization
# Paper Section 3.1, Equation 14
# ============================================================
def l2_normalize(X):
    """L2-normalize each row to unit length (Eq. 14)."""
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return X / norms


# Compute ALL 4 normalization variants for ablation
configs = {
    'Standard FK (no norm)': (train_fv, test_fv),
    'Power norm only':       (power_normalize(train_fv), power_normalize(test_fv)),
    'L2 norm only':          (l2_normalize(train_fv), l2_normalize(test_fv)),
    'IFK (Power + L2)':      (l2_normalize(power_normalize(train_fv)), 
                              l2_normalize(power_normalize(test_fv))),
}

print("All 4 normalization variants computed:")
for name in configs:
    print(f"  ✓ {name}")

All 4 normalization variants computed:
  ✓ Standard FK (no norm)
  ✓ Power norm only
  ✓ L2 norm only
  ✓ IFK (Power + L2)


L2 normalization divides the Fisher Vector by its L2 norm, projecting it onto the unit hypersphere (Eq. 14). As derived in Section 3.1 (Equations 9–13), the Fisher Vector magnitude depends on $\omega$ — the proportion of image-specific content. L2 normalization cancels this, making the representation robust to varying amounts of background. When combining both normalizations, power normalization is applied first (end of Section 3.2). We prepare all four variants (no norm, power only, L2 only, both) to mirror the paper's ablation in Table 1.

In [8]:
# ============================================================
# STEP 7: Linear SVM Classification + Evaluation
# Paper Section 4.1: "We learn linear SVMs with a hinge loss
# using the primal formulation and SGD."
# Metric: mAP (Section 4.2)
# ============================================================

# Binarize labels for mAP computation
classes = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)

results = {}

for name, (X_tr, X_te) in configs.items():
    svm = LinearSVC(
        C=SVM_C,
        max_iter=SVM_MAX_ITER,
        random_state=RANDOM_SEED,
        dual=True
    )
    svm.fit(X_tr, y_train)
    
    y_pred = svm.predict(X_te)
    scores = svm.decision_function(X_te)
    
    mAP = average_precision_score(y_test_bin, scores, average='macro')
    accuracy = accuracy_score(y_test, y_pred)
    
    results[name] = {'mAP': mAP, 'accuracy': accuracy}
    print(f"{name:25s} → mAP: {mAP:.4f}  |  Accuracy: {accuracy:.2%}")

# Save results for Task 2.3

print(f"\n--- Paper's results on PASCAL VOC 2007 (SIFT, K=256) ---")
print(f"Standard FK (no norm): 47.9% AP")
print(f"Power norm only:       54.2% AP  (+6.3%)")
print(f"L2 norm only:          51.8% AP  (+3.9%)")
print(f"IFK (Power + L2):      55.3% AP  (+7.4%)")

Standard FK (no norm)     → mAP: 0.8640  |  Accuracy: 83.50%


Power norm only           → mAP: 0.7560  |  Accuracy: 80.50%
L2 norm only              → mAP: 0.8710  |  Accuracy: 83.00%


IFK (Power + L2)          → mAP: 0.8762  |  Accuracy: 84.50%

--- Paper's results on PASCAL VOC 2007 (SIFT, K=256) ---
Standard FK (no norm): 47.9% AP
Power norm only:       54.2% AP  (+6.3%)
L2 norm only:          51.8% AP  (+3.9%)
IFK (Power + L2):      55.3% AP  (+7.4%)


We train linear SVMs on all four normalization variants, mirroring the ablation in Table 1 of the paper. The SVM uses a hinge loss in the primal formulation (Section 4.1). We compute mAP — the same metric as the paper — by getting per-class Average Precision from the SVM's decision function scores.

### Interpretation

The key thing to check is whether the **IFK (Power + L2)** outperforms the **Standard FK (no norm)** baseline, and whether the trend matches the paper's observation that power normalization contributes the most individually. The absolute numbers will differ significantly (different dataset, descriptors, K), but the *relative ordering* should demonstrate the paper's core claim: these normalizations make the Fisher Vector more effective with linear classifiers.